# ITI Retail — Prepare the Combined Data Review
This notebook follows the uploaded audit report. It resolves known folder-name aliases, confirms that 3-body images add no new file bytes, and prepares small copies of all multi images for visual review.

**No model training or automatic crop labels are applied.** The purpose is to finish data decisions before training. Original ZIP files, existing models, and original images remain unchanged. CPU is sufficient. All five sources remain accounted for: 100x100 as baseline, original-size as candidate detail/data source, 3-body as a split/overlap reference, multi for realistic images, and meta as reference information already exported in the audit.

The audit found 49,236 3-body image files, all matched by SHA-256 to files in 100x100. Repeating them would increase file count, not independent information.

## 1. Open the saved audit
Use the exact audit folder that produced your uploaded report. If your folder differs, edit only audit_path. Output is placed in a new review folder.

In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime
import json
import zipfile
import re
import pandas as pd
from PIL import Image, ImageOps
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/ITI_Retail_Project")
audit_path = project / "dataset_audit" / "20260906_144352"
assert (audit_path / "image_inventory.csv").is_file(), "Check audit_path."
images = pd.read_csv(audit_path / "image_inventory.csv").fillna("")
duplicates = pd.read_csv(audit_path / "exact_duplicates_across_sources.csv")
output_path = project / "data_review" / datetime.now().strftime("%Y%m%d_%H%M%S")
output_path.mkdir(parents=True, exist_ok=False)
preview_path = output_path / "multi_previews"
preview_path.mkdir()
print("Review folder:", output_path)

Mounted at /content/drive
Review folder: /content/drive/MyDrive/ITI_Retail_Project/data_review/20260906_150238


## 2. Resolve known folder spelling differences
Spaces, underscores, and letter case are normalized for lookup. One documented spelling difference is handled explicitly. Pear common 1 maps to the general Pear category without claiming it is the same physical fruit as Pear 1.

A name mapping is not a visual identity check and does not prove that two image files show the same object.

In [ ]:
def normalize_name(name):
    name = " ".join(name.lower().replace("_", " ").split())
    return name.replace("apple red delicios", "apple red delicious")

mapping_path = project / "general_categories" / "models" / "category_mapping.json"
old_mapping = json.loads(mapping_path.read_text())
lookup = {}
for name, category in old_mapping.items():
    key = normalize_name(name)
    assert key not in lookup or lookup[key] == category, "Conflicting label aliases."
    lookup[key] = category
for category in set(old_mapping.values()):
    lookup[normalize_name(category)] = category
lookup["pear common 1"] = "Pear"
lookup.update({"apples": "Apple", "cherries": "Cherry", "tomatoes": "Tomato"})

label_rows = []
for (source, name), count in images.groupby(["source", "source_label"]).size().items():
    category = lookup.get(normalize_name(name), "") if name else ""
    label_rows.append({"source": source, "original_label": name,
                       "general_category": category, "images": count,
                       "status": "Mapped" if category else "Visual review required"})
label_table = pd.DataFrame(label_rows)
label_table.to_csv(output_path / "resolved_label_mapping.csv", index=False)
structured = label_table[label_table["source"] != "multi"]
assert (structured["general_category"] != "").all(), "An additional structured-source label needs review."
display(label_table[label_table["status"] != "Mapped"])
print("All structured-source labels mapped to the existing general categories.")

,source,original_label,general_category,images,status
267,multi,,,756,Visual review required


All structured-source labels mapped to the existing general categories.


## 3. Record duplicate evidence and original-size matching candidates
SHA-256 confirms only the duplicate groups checked by the previous notebook. Matching a normalized variety name and frame filename between 100x100 and original-size creates a review candidate, not a confirmed duplicate. Keep ambiguous name/frame keys flagged.

These matches must not be blindly used to delete images or claim independent splits. Related recording frames and recompressed images need review before final split assignment.

In [ ]:
baseline_hashes = set(duplicates[duplicates["source"] == "100x100"]["sha256"])
three_duplicates = duplicates[duplicates["source"] == "3-body-problem"]
three_count = int((images["source"] == "3-body-problem").sum())
three_matched = int(three_duplicates["sha256"].isin(baseline_hashes).sum())
print("3-body image files:", three_count)
print("3-body files identical to 100x100:", three_matched)

paired = images[images["source"].isin(["100x100", "original-size"])].copy()
keys = []
for row in paired.to_dict("records"):
    stem = Path(row["member"]).stem.lower()
    if row["source"] == "100x100" and stem.endswith("_100"):
        stem = stem[:-4]
    keys.append(normalize_name(row["source_label"]) + "/" + stem)
paired["candidate_key"] = keys
counts = paired.groupby(["candidate_key", "source"]).size().unstack(fill_value=0)
matched_keys = counts[(counts["100x100"] > 0) & (counts["original-size"] > 0)].index
candidates = paired[paired["candidate_key"].isin(matched_keys)]
candidates.to_csv(output_path / "name_frame_candidates_NOT_verified.csv", index=False)
ambiguous = counts[(counts > 1).any(axis=1)]
ambiguous.to_csv(output_path / "ambiguous_name_frame_keys.csv")
print("Original-size files with a name/frame candidate:", int(((paired["source"] == "original-size") & paired["candidate_key"].isin(matched_keys)).sum()))
print("These are candidate correspondences, not confirmed image duplicates.")

3-body image files: 49236
3-body files identical to 100x100: 49236
Original-size files with a name/frame candidate: 94263
These are candidate correspondences, not confirmed image duplicates.


## 4. Prepare all multi images for review
Create lightweight JPEG previews with a maximum side of 800 pixels and a stable ID. The manifest keeps each original archive path, image size, EXIF time when available, and filename hints. Hints are not accepted class labels.

This reads the 756 original files from Drive, so allow time to finish. Images are not cropped, labeled, or placed into training. The previews allow review without uploading the multi-gigabyte original archive. Training crops will later be extracted from the original files, not these previews.

In [ ]:
multi_path = project / "fruits-360-multi-main.zip"
assert multi_path.is_file(), "multi archive not found."
multi_table = images[images["source"] == "multi"].sort_values("member")
review_rows = []
with zipfile.ZipFile(multi_path) as archive:
    for index, row in enumerate(multi_table.to_dict("records"), start=1):
        image_id = "multi_" + str(index).zfill(4)
        record = {"image_id": image_id, "original_member": row["member"],
                  "filename_hint": Path(row["member"]).name, "preview_file": "",
                  "capture_time": "", "reviewed_label": "", "capture_group": "",
                  "review_status": "Pending visual review", "error": ""}
        try:
            with Image.open(BytesIO(archive.read(row["member"]))) as original:
                record["original_width"], record["original_height"] = original.size
                exif = original.getexif()
                record["capture_time"] = str(exif.get(36867, exif.get(306, "")))
                image = ImageOps.exif_transpose(original).convert("RGB")
                record["oriented_width"], record["oriented_height"] = image.size
                image.thumbnail((800, 800))
                name = image_id + ".jpg"
                image.save(preview_path / name, quality=85)
                record["preview_file"] = "multi_previews/" + name
        except Exception as error:
            record["error"] = str(error)
            record["review_status"] = "Read error"
        review_rows.append(record)
        if index % 50 == 0:
            print("Prepared", index, "of", len(multi_table))
review = pd.DataFrame(review_rows)
review.to_csv(output_path / "multi_review_manifest.csv", index=False)
print("Multi images:", len(review))
print("Read errors:", int((review["error"] != "").sum()))
display(review.head())

Prepared 50 of 756
Prepared 100 of 756
Prepared 150 of 756
Prepared 200 of 756
Prepared 250 of 756
Prepared 300 of 756
Prepared 350 of 756
Prepared 400 of 756
Prepared 450 of 756
Prepared 500 of 756
Prepared 550 of 756
Prepared 600 of 756
Prepared 650 of 756
Prepared 700 of 756
Prepared 750 of 756
Multi images: 756
Read errors: 0


,image_id,original_member,filename_hint,preview_file,capture_time,reviewed_label,capture_group,review_status,error,original_width,original_height,oriented_width,oriented_height
0,multi_0001,fruits-360-multi-main/test-multiple_fruits/Ban...,Banana(lady_finger)_1.jpg,multi_previews/multi_0001.jpg,2018:06:16 09:35:23,,,Pending visual review,,3024,4032,3024,4032
1,multi_0002,fruits-360-multi-main/test-multiple_fruits/Ban...,Banana(lady_finger)_2.jpg,multi_previews/multi_0002.jpg,2018:06:16 09:35:36,,,Pending visual review,,3024,4032,3024,4032
2,multi_0003,fruits-360-multi-main/test-multiple_fruits/Ban...,Banana(lady_finger)_3.jpg,multi_previews/multi_0003.jpg,2018:06:16 09:35:46,,,Pending visual review,,4032,3024,4032,3024
3,multi_0004,fruits-360-multi-main/test-multiple_fruits/Ban...,Banana(lady_finger)_4.jpg,multi_previews/multi_0004.jpg,2018:06:16 09:35:58,,,Pending visual review,,4032,3024,4032,3024
4,multi_0005,fruits-360-multi-main/test-multiple_fruits/Ban...,Banana(lady_finger)_5.jpg,multi_previews/multi_0005.jpg,2018:06:16 09:36:20,,,Pending visual review,,4032,3024,4032,3024


## 5. Export the review pack
Send the generated ZIP. It contains all multi previews and the review tables. It does not modify the current application.

Next: review usable products and crop coordinates, identify related source-photo groups, resolve image correspondences, and freeze splits. All crops and image variants from one original photo or capture group must stay together. Unknown products in multi are not automatically forced into the current 84 classes.

In [ ]:
summary = {
    "3_body_total_files": three_count,
    "3_body_exact_matches_to_100x100": three_matched,
    "multi_total": len(review),
    "multi_preview_errors": int((review["error"] != "").sum()),
    "structured_labels_resolved": bool((structured["general_category"] != "").all()),
    "ready_for_training": False
}
(output_path / "review_summary.json").write_text(json.dumps(summary, indent=2))
export_path = output_path.parent / (output_path.name + "_Data_Review.zip")
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in output_path.rglob("*"):
        if path.is_file():
            archive.write(path, str(path.relative_to(output_path)))
print("REVIEW PACK COMPLETE")
print(json.dumps(summary, indent=2))
print("Send this file:", export_path)

REVIEW PACK COMPLETE
{
  "3_body_total_files": 49236,
  "3_body_exact_matches_to_100x100": 49236,
  "multi_total": 756,
  "multi_preview_errors": 0,
  "structured_labels_resolved": true,
  "ready_for_training": false
}
Send this file: /content/drive/MyDrive/ITI_Retail_Project/data_review/20260906_150238_Data_Review.zip
